# Fine-tuning PaddleOCR Recognition (Step 2) trên Manga109-s

Pipeline đầy đủ:

1. **Cài đặt môi trường** – PaddlePaddle-GPU + PaddleOCR
2. **Tải dataset** – từ Hugging Face hoặc Kaggle Dataset
3. **Tạo REC dataset** – crop text box, tạo label files (train/val split)
4. **Tạo dictionary** – trích ký tự từ label files
5. **Tải pretrained weights** – Japan PP-OCRv3 Rec
6. **Tạo config YAML** – auto-resume nếu có checkpoint
7. **Training** – distributed 2× GPU (T4)


## 1. Cài đặt môi trường

In [1]:
import sys, subprocess, os, importlib, re

In [2]:
!git clone https://github.com/KhoaLeDang2375/crnn.pytorch.git
%cd crnn.pytorch

Cloning into 'crnn.pytorch'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 199 (delta 98), reused 99 (delta 53), pack-reused 47 (from 1)
Receiving objects: 100% (199/199), 67.06 KiB | 2.48 MiB/s, done.
Resolving deltas: 100% (107/107), done.
/kaggle/working/crnn.pytorch


## 2. Tải dataset Manga109-s

> Chạy cell này **một lần duy nhất**. Nếu đã có file zip sẵn, bỏ qua.


In [ ]:
import os, json

HF_TOKEN = '' 
BASE_DIR  = '/kaggle/working/manga109s_dataset'
os.makedirs(BASE_DIR, exist_ok=True)

# Dọn cache cũ để tránh lỗi symlink
import subprocess
subprocess.run(['rm', '-rf',
    '/root/.cache/huggingface/datasets',
    '/kaggle/working/hf_cache'
], check=False)

# Tải dataset từ Hugging Face
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
from huggingface_hub import login, snapshot_download
login(HF_TOKEN)

print('📥 Đang tải Manga109-s từ Hugging Face...')
snapshot_download(
    repo_id='hal-utokyo/Manga109-s',
    repo_type='dataset',
    local_dir=BASE_DIR,
    token=HF_TOKEN,
)
print(f'✅ Tải xong. Files: {os.listdir(BASE_DIR)}')


📥 Đang tải Manga109-s từ Hugging Face...


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Tải xong. Files: ['Manga109s_released_2023_12_07.zip', '.gitattributes', '.cache', 'README.md']


## 3. Giải nén dataset

In [4]:
import zipfile, os

ZIP_PATH     = '/kaggle/working/manga109s_dataset/Manga109s_released_2023_12_07.zip'
EXTRACT_PATH = '/kaggle/working/manga109_extracted'
DATASET_ROOT = f'{EXTRACT_PATH}/Manga109s_released_2023_12_07'

if not os.path.exists(DATASET_ROOT) or not os.listdir(DATASET_ROOT):
    print('📦 Đang giải nén (~3 GB)...')
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_PATH)
    print('✅ Giải nén xong!')
else:
    print('✅ Đã giải nén từ trước, bỏ qua.')

print(f'Nội dung: {os.listdir(DATASET_ROOT)}')


📦 Đang giải nén (~3 GB)...
✅ Giải nén xong!
Nội dung: ['images', 'annotations_Manga109Dialog', 'annotations.v2018.05.31', 'books.txt', 'annotations.v2020.12.18', 'annotations_COO', 'readme.txt', 'annotations']


## 4. Tạo REC dataset (crop text box)

Duyệt toàn bộ annotation XML, crop từng text box thành ảnh nhỏ,
lưu kèm label file theo định dạng PaddleOCR.

**Cấu trúc output:**
```
train_data/rec/
├── train/          ← ảnh crop (word_000001.png ...)
├── val/
├── rec_gt_train.txt
└── rec_gt_val.txt
```


In [5]:
import os, random
import xml.etree.ElementTree as ET
from PIL import Image
from typing import Optional
from tqdm import tqdm


class Manga109RecDatasetCreator:
    """
    Tạo dataset Text Recognition (PaddleOCR format) từ Manga109-s.
    Crop từng text box → ảnh riêng, kèm label file train/val.
    """

    def __init__(self, dataset_root: str):
        self.dataset_root    = dataset_root
        self.annotations_dir = os.path.join(dataset_root, 'annotations')
        self.images_dir      = os.path.join(dataset_root, 'images')

    def create_rec_dataset(
        self,
        output_dir:  str = '/kaggle/working/train_data',
        min_width:   int = 20,
        min_height:  int = 20,
        max_samples: Optional[int] = 20_000,
        train_ratio: float = 0.9,
        seed:        int = 42,
    ):
        random.seed(seed)

        rec_dir       = os.path.join(output_dir, 'rec')
        train_img_dir = os.path.join(rec_dir, 'train')
        val_img_dir   = os.path.join(rec_dir, 'val')
        os.makedirs(train_img_dir, exist_ok=True)
        os.makedirs(val_img_dir,   exist_ok=True)

        train_gt = os.path.join(rec_dir, 'rec_gt_train.txt')
        val_gt   = os.path.join(rec_dir, 'rec_gt_val.txt')

        xml_files = sorted(f for f in os.listdir(self.annotations_dir) if f.endswith('.xml'))
        print(f'🔄 Tìm thấy {len(xml_files)} truyện. Đang thu thập samples...')

        # Thu thập tất cả samples trước để shuffle đều
        all_samples = []  # list of (PIL.Image crop, label str)

        for xml_file in tqdm(xml_files, desc='Parsing books'):
            book_title = os.path.splitext(xml_file)[0]
            xml_path   = os.path.join(self.annotations_dir, xml_file)
            try:
                root = ET.parse(xml_path).getroot()
            except Exception:
                continue

            for page in root.findall('.//page'):
                page_idx     = page.get('index')
                img_filename = f'{int(page_idx):03d}.jpg'
                img_path     = os.path.join(self.images_dir, book_title, img_filename)
                if not os.path.exists(img_path):
                    continue
                try:
                    img = Image.open(img_path).convert('RGB')
                except Exception:
                    continue

                for text_obj in page.findall('text'):
                    try:
                        xmin  = int(text_obj.get('xmin'))
                        ymin  = int(text_obj.get('ymin'))
                        xmax  = int(text_obj.get('xmax'))
                        ymax  = int(text_obj.get('ymax'))
                        label = (text_obj.text or '').strip()
                        label = label.replace('\n', '').replace('\t', ' ')
                        if not label:
                            continue
                        if (xmax - xmin) < min_width or (ymax - ymin) < min_height:
                            continue
                        all_samples.append((img.crop((xmin, ymin, xmax, ymax)), label))
                    except Exception:
                        continue

            # Dừng sớm khi đã đủ mẫu (thu dư 2x để shuffle chắc)
            if max_samples and len(all_samples) >= max_samples * 2:
                break

        # Giới hạn & shuffle
        random.shuffle(all_samples)
        if max_samples:
            all_samples = all_samples[:max_samples]

        split_idx     = int(len(all_samples) * train_ratio)
        train_samples = all_samples[:split_idx]
        val_samples   = all_samples[split_idx:]
        print(f'\n📊 Split: {len(train_samples):,} train | {len(val_samples):,} val')

        def _write_split(samples, img_dir, gt_path, split_name, start_idx=1):
            with open(gt_path, 'w', encoding='utf-8') as gt_file:
                for i, (crop, label) in enumerate(
                    tqdm(samples, desc=f'Saving {split_name}'), start=start_idx
                ):
                    img_name  = f'word_{i:06d}.png'
                    save_path = os.path.join(img_dir, img_name)
                    try:
                        crop.save(save_path)
                    except Exception:
                        continue
                    gt_file.write(f'{split_name}/{img_name}\t{label}\n')
            return len(samples)

        n_train = _write_split(train_samples, train_img_dir, train_gt, 'train', start_idx=1)
        n_val   = _write_split(val_samples,   val_img_dir,   val_gt,   'val',   start_idx=n_train + 1)

        print(f'\n✅ HOÀN THÀNH Recognition Dataset!')
        print(f'   Train : {n_train:,} ảnh  →  {train_gt}')
        print(f'   Val   : {n_val:,} ảnh  →  {val_gt}')


creator = Manga109RecDatasetCreator(DATASET_ROOT)
creator.create_rec_dataset(
    output_dir  = '/kaggle/working/train_data',
    max_samples = 50_000,
    train_ratio = 0.9,
    seed        = 42,
)


🔄 Tìm thấy 87 truyện. Đang thu thập samples...



Parsing books:  97%|█████████▋| 84/87 [01:33<00:03,  1.12s/it]



📊 Split: 45,000 train | 5,000 val



Saving train: 100%|██████████| 45000/45000 [01:49<00:00, 410.99it/s]

Saving val: 100%|██████████| 5000/5000 [00:12<00:00, 409.01it/s]



✅ HOÀN THÀNH Recognition Dataset!
   Train : 45,000 ảnh  →  /kaggle/working/train_data/rec/rec_gt_train.txt
   Val   : 5,000 ảnh  →  /kaggle/working/train_data/rec/rec_gt_val.txt


## 5. Cấu hình đường dẫn training

In [6]:
import os

DATA_ROOT     = '/kaggle/working/train_data'
PRETRAIN_DIR  = '/kaggle/working/pretrained'
OUTPUT_DIR    = '/kaggle/working/output/rec_ppocr_manga109'
PADDLEOCR_DIR = '/kaggle/working/PaddleOCR'
DICT_PATH     = '/kaggle/working/manga109_dict.txt'

os.makedirs(PRETRAIN_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,   exist_ok=True)

# Kiểm tra data tồn tại
for f in [
    f'{DATA_ROOT}/rec/rec_gt_train.txt',
    f'{DATA_ROOT}/rec/rec_gt_val.txt',
]:
    if os.path.exists(f):
        with open(f) as fp:
            n = sum(1 for _ in fp)
        print(f'✅ {os.path.basename(f):30s} → {n:,} dòng')
    else:
        print(f'❌ KHÔNG TÌM THẤY: {f}')


✅ rec_gt_train.txt               → 45,000 dòng
✅ rec_gt_val.txt                 → 5,000 dòng


## 6. Tạo character dictionary

In [7]:
!python tool/extract_alphabet.py \
    --train_txt "/kaggle/working/train_data/rec/rec_gt_train.txt" \
    --val_txt "/kaggle/working/train_data/rec/rec_gt_val.txt" \
    --out_dict "/kaggle/working/dict.txt" 

Trích xuất thành công 2876 ký tự duy nhất.
Lưu Bộ từ vựng vào: /kaggle/working/dict.txt


In [8]:
!pip install lmdb editdistance

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.9/325.9 kB 6.1 MB/s eta 0:00:00a 0:00:01


In [9]:
!python tool/create_lmdb_dataset.py \
    --train_txt "/kaggle/working/train_data/rec/rec_gt_train.txt" \
    --val_txt "/kaggle/working/train_data/rec/rec_gt_val.txt" \
    --img_dir "/kaggle/working/train_data/rec/" \
    --out_dir "/kaggle/working/lmdb_dataset"


Đang tạo LMDB cho tập Train...
Written 1000 / 45000
Written 2000 / 45000
Written 3000 / 45000
Written 4000 / 45000
Written 5000 / 45000
Written 6000 / 45000
Written 7000 / 45000
Written 8000 / 45000
Written 9000 / 45000
Written 10000 / 45000
Written 11000 / 45000
Written 12000 / 45000
Written 13000 / 45000
Written 14000 / 45000
Written 15000 / 45000
Written 16000 / 45000
Written 17000 / 45000
Written 18000 / 45000
Written 19000 / 45000
Written 20000 / 45000
Written 21000 / 45000
Written 22000 / 45000
Written 23000 / 45000
Written 24000 / 45000
Written 25000 / 45000
Written 26000 / 45000
Written 27000 / 45000
Written 28000 / 45000
Written 29000 / 45000
Written 30000 / 45000
Written 31000 / 45000
Written 32000 / 45000
Written 33000 / 45000
Written 34000 / 45000
Written 35000 / 45000
Written 36000 / 45000
Written 37000 / 45000
Written 38000 / 45000
Written 39000 / 45000
Written 40000 / 45000
Written 41000 / 45000
Written 42000 / 45000
Written 43000 / 45000
Written 44000 / 45000
Written 45

## 7. Tải pretrained weights

In [10]:
!python train.py \
  --trainRoot /kaggle/working/lmdb_dataset/train \
  --valRoot /kaggle/working/lmdb_dataset/val \
  --batchSize 256 \
  --ngpu 1 \
  --nepoch 100 \
  --imgH 32 \
  --imgW 256 \
  --dict /kaggle/working/dict.txt \
  --cuda \
  --workers 4 \
  --printEvery 50 \
  --adadelta

Namespace(trainRoot='/kaggle/working/lmdb_dataset/train', valRoot='/kaggle/working/lmdb_dataset/val', workers=4, batchSize=256, imgH=32, imgW=256, nh=256, nepoch=100, printEvery=50, valEvery=500, cuda=True, ngpu=1, pretrained='', alphabet=' !"%&\'()+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\\]^_abcdefghijklmnoprstuvwxyz~×ΛαβθμξτВЕавдеиклморсчя―‘’“”†•‥…※‼⁉℃ℓⅡⅣ←↑→↓↘⇒−∞∠≡≪≫⊃⊕①②③④⑤⑦■□△▼◀○●◯★☆☓☺♡♥♪♫♬✡✩❀⭐\u3000、。〃々〆〇〈〉《》「」『』〜〝〟〰〱ぁあぃいぅうぇえぉおかがきぎくぐけげこごさざしじすずせぜそぞただちぢっつづてでとどなにぬねのはばぱひびぴふぶぷへべぺほぼぽまみむめもゃやゅゆょよらりるれろわゐをんゔ゙゛゜ゝァアィイゥウェエォオカガキギクグケゲコゴサザシジスズセゼソゾタダチヂッツヅテデトドナニヌネノハバパヒビピフブプヘベペホボポマミムメモャヤュユョヨラリルレロワヰヲンヴヵヶ・ーㇸ㊟㋬㌧㎏㎝㎞一丁七万丈三上下不与丑世丘丞両並中串丸丹主丼乃久之乏乗乙九乞也乱乳乾亀了予争事二云互五井亘亜亡交亦享京亭亮人仁仇今介仏仔仕他付仙代令以仮仰仲件任企伊伍伎伏伐休会伝伯伴伸伺似但佇位低住佐佑体何余作佳併使來例侍供依侠価侮侵便係俊俗保俠信修俳俸俺倉個倍倒倖候借値倫倶偉偏停健側偵偶偽傍傑傘備催傭傲傷傾働像僕僚僧僻儀億儒儚償優儲元兄充先光克免児兒党兜入全八公六共兵其具典兼内円冊再冒冗写冠冥冬冴冶冷凄凍凝几凡処凧凰凶凸凹出函刀刃分切刈刊刑列初判別利到制券刹刺刻剃則削前剖剛剝剣剤剥副剰割創劇力功加劣助努励労効劾勃勇勉動勘務勝募勢勤勧勲勾勿匂包化北匠匹区医匿十千午半卑卒卓協南単博卜占卯印危即却卵卿厄厚原厠厨厳去参又及友双反収叔取受叙口古句叩只叫召可台叱史右叶号司各合吉吊同名后吐向君吟吠否含吸吹吼吾呂呆呈呉告呑周呪味呼命咆和咲哀品員哨哮哲唄唇唐唯唱商問啓善喉喋喘喚喜喧喩喪喫喰営嗅嗚

In [11]:
!ls -lh expr/

total 21M
-rw-r--r-- 1 root root 21M Mar 25 14:05 netCRNN_best.pth
